# ISIC synthetic-hair study — POST-TRAINING ANALYSIS NOTEBOOK

Run this only after the training notebook printed `ALL 30 MODELS COMPLETE` and you committed that version.

**Before running, in the right panel:**
- Accelerator: GPU T4 x2 | Internet: On | Persistence: Variables and Files
- **Add Data:**
  1. your project dataset (same one used for training)
  2. `isic-2019-dataset-full`
  3. **the TRAINING notebook's own committed output** (Add Data → Your Work → the 30/30 version)

Run cells top to bottom. Stage 0/0b/1 are fast (~15–30 min total, mostly the cache rebuild). Stages 2 (extraction) and 3 (Integrated Gradients) are the GPU-heavy parts and are resumable exactly like the training notebook — set `MAX_MODELS_THIS_RUN` and use Save & Run All (Commit) with the same chunking approach if needed. **Stage 4 only runs once Stages 2 and 3 both show 30/30** — it computes no endpoint before every model has contributed.


### Cell 1 — config
`MAX_MODELS_THIS_RUN` and `DEADLINE_HOURS` control this run's scope, same pattern as the training notebook. Set `MAX_MODELS_THIS_RUN` low for a first smoke-test commit before committing to the full 30.

In [1]:
# ================================================================
# CELL 1 — CONFIG (ANALYSIS NOTEBOOK)
# Add Data (right panel):
#   (1) your project dataset (same one used for training)                      -> gives the locked project + evaluation manifests + XAI subset
#   (2) isic-2019-dataset-full                                                 -> needed to rebuild the 224x224 cache
#   (3) the TRAINING notebook's own output (Add Data -> Your Work -> that version) -> gives training_runs/ (30 checkpoints) + kaggle_records/ (GAP_DECISION.json etc.)
# Settings: Accelerator = GPU T4 x2 | Internet = ON | Persistence = Variables and Files
# ================================================================
AUTO_PIN_VERSIONS = globals().get("AUTO_PIN_VERSIONS", True)
ALLOW_VERSION_DEVIATION = globals().get("ALLOW_VERSION_DEVIATION", False)
N_PROC_OVERRIDE = globals().get("N_PROC_OVERRIDE", 4)
MAX_MODELS_THIS_RUN = globals().get("MAX_MODELS_THIS_RUN", None)   # cap how many of the 30 models get processed this run (for resuming across commits)
DEADLINE_HOURS = globals().get("DEADLINE_HOURS", 11.0)
EST_MODEL_HOURS = 0.15   # scheduling estimate per model for extraction+IG combined (~9 min) - revised after the first model's real timing is known

print("✓ analysis-notebook config loaded")


✓ analysis-notebook config loaded


### Cell 2 — helpers
Same tested path/discovery logic as the training notebook (layout-independent search under /kaggle/input).

In [2]:
# ================================================================
# CELL 2 — HELPERS (paths, hashing, layout-independent discovery - same tested logic as the training notebook)
# ================================================================
import os, sys, json, time, shutil, hashlib, subprocess, platform
from pathlib import Path
from collections import deque
from datetime import datetime, timezone

# ---------------- paths ----------------
# Kaggle's dataset mount point is NOT fixed: it has been seen both as
#   /kaggle/input/<dataset-slug>/...                              (older / some accounts)
#   /kaggle/input/datasets/<owner>/<dataset-slug>/...             (this account)
# and can occasionally double-nest (.../<slug>/<slug>/...). Rather than hard-code either layout, every path below
# is found by SEARCHING under /kaggle/input for a directory that satisfies a content-based test (see Stage 0/2),
# so the code works regardless of which layout Kaggle gives you.
ROOT = Path(os.environ.get("KAGGLE_TEST_ROOT", "/kaggle"))
INPUT, WORK = ROOT / "input", ROOT / "working"
TEMP = ROOT / "temp"
TEMP.mkdir(parents=True, exist_ok=True); WORK.mkdir(parents=True, exist_ok=True)
PROJECT = WORK / "isic_final_project"          # working copy of the locked project (protocol, evaluation manifests, XAI subset, frozen analysis code)
CACHE = TEMP / "isic_cache"                    # 224x224 image cache (rebuilt fresh this session, NOT saved as output)
RECORDS = WORK / "analysis_records"            # this notebook's own small records - saved as output
RESULTS = WORK / "results"                     # per-model endpoints, IG results, final statistics - saved as output
TRAINED_MODELS = WORK / "trained_models_readonly"   # copy of the training notebook's training_runs/ (read, never written)
LOCAL_TMP = TEMP / "tmp_work"
for d in (RECORDS, RESULTS, LOCAL_TMP): d.mkdir(parents=True, exist_ok=True)
T0 = time.time()

def utc(): return datetime.now(timezone.utc).isoformat()
def elapsed_h(): return (time.time() - T0) / 3600
def log(msg): print(f"[{elapsed_h():5.2f} h] {msg}", flush=True)
def sha_file(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()
def J(rel): return json.load(open(PROJECT / rel))
def save_record(name, obj):
    with open(RECORDS / name, "w") as f: json.dump(obj, f, indent=2, sort_keys=True, default=str)

# ---------------- content-based directory discovery (layout-independent) ----------------
def find_by_markers(root, required_subdirs, max_depth=6):
    """BFS under root for the shallowest directory whose immediate children include every name in required_subdirs.
    Skips hidden directories. Never reads file contents, so it is cheap even under a big dataset tree."""
    root = Path(root)
    if not root.exists(): return None
    required = set(required_subdirs)
    q = deque([(root, 0)])
    while q:
        d, depth = q.popleft()
        try:
            entries = list(os.scandir(d))
        except (PermissionError, FileNotFoundError, NotADirectoryError):
            continue
        names = {e.name for e in entries if e.is_dir()}
        if required <= names:
            return d
        if depth < max_depth:
            for e in entries:
                if e.is_dir() and not e.name.startswith("."):
                    q.append((Path(e.path), depth + 1))
    return None

def _has_min_files(d, exts, min_count, scan_cap=20000):
    n = 0
    try:
        with os.scandir(d) as it:
            for i, e in enumerate(it):
                if i >= scan_cap: break
                if e.is_file() and e.name.lower().endswith(exts):
                    n += 1
                    if n >= min_count: return True
    except (PermissionError, FileNotFoundError, NotADirectoryError):
        return False
    return False

def find_image_dir(root, exts=(".jpg", ".jpeg"), min_images=100, max_depth=6):
    """BFS under root for a directory that directly contains at least min_images image files.
    Does not descend into a directory once it qualifies (image folders are leaves), so this stays fast
    even on a 23k-image dataset. If several qualify, prefers one with 'train' in its path."""
    root = Path(root)
    if not root.exists(): return None
    q = deque([(root, 0)]); found = []
    while q:
        d, depth = q.popleft()
        try:
            entries = list(os.scandir(d))
        except (PermissionError, FileNotFoundError, NotADirectoryError):
            continue
        if _has_min_files(d, exts, min_images):
            found.append(d); continue
        if depth < max_depth:
            for e in entries:
                if e.is_dir() and not e.name.startswith("."):
                    q.append((Path(e.path), depth + 1))
    if not found: return None
    found.sort(key=lambda p: (0 if "train" in str(p).lower() else 1, len(str(p))))
    return found[0]



### Stage 0 — restore project, verify lock chain, locate the 30 trained models
Re-verifies every hash in the lock chain, then finds the training notebook's output dataset and re-checks each model's checkpoint hashes against its own training_record.json - the same verification the training notebook did before marking a model complete, done again here independently.

In [3]:
# ================================================================
# STAGE 0 — RESTORE THE LOCKED PROJECT + LOCATE THE TRAINED MODELS
# ================================================================
PROJECT_MARKERS = {"step0I_locked_clean_population", "step1F_lesion_level_rho_assignments"}

def _search(c):
    if not c.exists(): return None
    for base in [c] + [x for x in c.iterdir() if x.is_dir()]:
        if (base / "step0I_locked_clean_population").exists() and (base / "step1F_lesion_level_rho_assignments").exists():
            return base
    return None

if not PROJECT.exists():
    src = find_by_markers(INPUT, PROJECT_MARKERS)
    assert src is not None, ("Project folder not found anywhere under /kaggle/input (searched up to 6 levels deep for a "
        f"directory containing {sorted(PROJECT_MARKERS)}). Check the project dataset is attached (Add Data).")
    log(f"found locked project at {src} - copying (read-only source, nothing there is modified) ...")
    shutil.copytree(src, PROJECT)

# ---- verify the lock chain (same checks as the training notebook)
checks = []
def check(name, ok, detail=""):
    checks.append((name, bool(ok))); print(("✓ " if ok else "✗ ") + name + (f"   {detail}" if detail else "")); assert ok, "LOCK CHAIN CHECK FAILED: " + name
sha = lambda rel: sha_file(PROJECT / rel)
def J(rel): return json.load(open(PROJECT / rel))
F = J("step1U_final_protocol_lock/FINAL_PROTOCOL_HASH.json")
check("final protocol == recorded hash", sha("step1U_final_protocol_lock/FINAL_LOCKED_EXPERIMENT_PROTOCOL.json") == F["protocol_sha256"])
check("seed manifest == recorded hash", sha("step1P_seed_manifest/seed_manifest.json") == F["seed_manifest_sha256"])
GEN_SHA = J("step1J_generator_recovery_lock/generator_recovery_lock.json")["generator_sha256"]
check("generator == recorded hash", sha("step1J_generator_recovery_lock/recovered_original_hair_generator.py") == GEN_SHA)
check("analysis code == frozen hash", sha("step2W_analysis_code_freeze/analysis_pipeline.py") == J("step2W_analysis_code_freeze/ANALYSIS_CODE_FINAL_HASH.json")["sha256"])
E = J("step2S_evaluation_manifest_lock/EVALUATION_MANIFEST_FINAL_HASH.json")
check("3 evaluation manifests == locked hashes", all(sha(f"step2R_production_evaluation_manifests/evaluation_{c}.csv") == h for c, h in E["manifest_sha256"].items()))
check("XAI subset == locked hash", sha("step2U_xai_subset_lock/LOCKED_XAI_SUBSET.csv") == J("step2U_xai_subset_lock/XAI_SUBSET_FINAL_HASH.json")["sha256"])
RO = J("step2V_seed_table_and_run_order/RUN_ORDER_FINAL_HASH.json")
check("run-order manifest == locked hash", sha("step2V_seed_table_and_run_order/RUN_ORDER_MANIFEST.json") == RO["sha256"])
RUN_ORDER = json.load(open(PROJECT / "step2V_seed_table_and_run_order/RUN_ORDER_MANIFEST.json"))["run_order"]
check("run order lists 30 models", len(RUN_ORDER) == 30)
ANALYSIS_MODULE = PROJECT / "step2W_analysis_code_freeze" / "analysis_pipeline.py"
print("\n✓ locked project verified.")

# ---- locate the training notebook's own output (added via Add Data -> Your Work)
TRAINING_MARKERS = {"training_runs", "kaggle_records"}
if not TRAINED_MODELS.exists():
    src = find_by_markers(INPUT, TRAINING_MARKERS)
    assert src is not None, ("Training output not found under /kaggle/input. Add the TRAINING notebook's own committed "
        "version as a data source (Add Data -> Your Work -> pick the version with 30/30 models complete).")
    log(f"found training output at {src} - copying (read-only source) ...")
    shutil.copytree(src, TRAINED_MODELS)

TR = TRAINED_MODELS / "training_runs"
TRAIN_RECORDS = TRAINED_MODELS / "kaggle_records"
model_dirs = sorted(TR.glob("rho_*_rep_*"))
print(f"training_runs: {len(model_dirs)} model folders found")
present = set()
for d in model_dirs:
    rec_p = d / "training_record.json"
    if not rec_p.exists(): continue
    r = json.load(open(rec_p))
    if r.get("status") == "COMPLETED" and all((d / f).exists() and sha_file(d / f) == h for f, h in r["checkpoints"].items()):
        present.add(d.name)
expected_names = [f"rho_{o['rho']:.1f}_rep_{o['replicate']}" for o in RUN_ORDER]
missing = [n for n in expected_names if n not in present]
assert not missing, f"{len(missing)} models missing or failed hash verification in the training output: {missing}"
print(f"✓ all 30 trained models present and hash-verified: {sorted(present) == sorted(expected_names)}")

GAP_DECISION = json.load(open(TRAIN_RECORDS / "GAP_DECISION.json"))["gap"]
print("pooling implementation used for all 30 models:", GAP_DECISION)
print("\nSTAGE 0 PASSED.")


[ 0.00 h] found locked project at /kaggle/input/datasets/tejasseshadriiiii/isic-final-project - copying (read-only source, nothing there is modified) ...
✓ final protocol == recorded hash
✓ seed manifest == recorded hash
✓ generator == recorded hash
✓ analysis code == frozen hash
✓ 3 evaluation manifests == locked hashes
✓ XAI subset == locked hash
✓ run-order manifest == locked hash
✓ run order lists 30 models

✓ locked project verified.
[ 0.01 h] found training output at /kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained - copying (read-only source) ...
training_runs: 30 model folders found
✓ all 30 trained models present and hash-verified: True
pooling implementation used for all 30 models: standard

STAGE 0 PASSED.


### Stage 0b — compatibility symlink
Same fix as the training notebook: the locked generator has a hard-coded Colab path baked in; this makes it resolve without touching the generator's bytes (hash re-verified after).

In [4]:
# ================================================================
# STAGE 0b — COMPATIBILITY SYMLINK  (same reason as the training notebook: the locked generator has a
# hard-coded /content/isic_final_project/... path baked in from Colab. We do NOT modify the generator - we
# make that path resolve to our real project copy, so the frozen bytes run unmodified.)
# ================================================================
CONTENT_PROJECT = Path("/content/isic_final_project")
if CONTENT_PROJECT.exists():
    print(f"✓ {CONTENT_PROJECT} already exists - leaving it as is")
else:
    CONTENT_PROJECT.parent.mkdir(parents=True, exist_ok=True)
    try:
        CONTENT_PROJECT.symlink_to(PROJECT, target_is_directory=True)
        print(f"✓ compatibility symlink created: {CONTENT_PROJECT} -> {PROJECT}")
    except OSError as e:
        print(f"Symlink failed ({e}) - falling back to a full copy")
        shutil.copytree(PROJECT, CONTENT_PROJECT)

gen_sha = sha_file(PROJECT / "step1J_generator_recovery_lock" / "recovered_original_hair_generator.py")
assert gen_sha == json.load(open(PROJECT / "step1J_generator_recovery_lock" / "generator_recovery_lock.json"))["generator_sha256"]
print("✓ generator file hash unchanged")


✓ compatibility symlink created: /content/isic_final_project -> /kaggle/working/isic_final_project
✓ generator file hash unchanged


In [5]:
# ================================================================
# CLEAN FAILED/PARTIAL KAGGLE CACHE
# ================================================================

from pathlib import Path

# Recreate the lock path used by Stage 2
CACHE_LOCK = RECORDS / "cache_final_hash.json"

for name in [
    "clean_224.npy",
    "hair_224.npy",
    "mask_area.npy",
    "cache_index.csv",
    "_builder_record.json",
]:
    p = CACHE / name
    if p.exists():
        p.unlink()
        print("removed:", p)

if CACHE_LOCK.exists():
    CACHE_LOCK.unlink()
    print("removed:", CACHE_LOCK)

print("✓ Partial cache removed. Ready for a clean rebuild.")

✓ Partial cache removed. Ready for a clean rebuild.


### Stage 1 — build the 224×224 cache
Needed again this session (the cache isn't part of any committed output - it's rebuilt fresh, same subprocess construction as training, so it's byte-identical).

In [6]:
# ================================================================
# STAGE 2 — BUILD THE 224x224 CACHE ON KAGGLE
#
# KAGGLE PLATFORM-SAFE VERSION
#
# IMPORTANT:
# - The scientific hair generator itself is NOT modified.
# - The recovered generator file also contains the original Step 1A
#   validation/test code at module level.
# - Executing that entire file inside multiprocessing workers causes
#   the Step 1A assertions to run repeatedly and crash the workers.
# - Therefore we extract ONLY:
#       * imports
#       * literal/global constants
#       * function definitions
#   from the recovered generator.
#
# This changes only HOW the locked generator is loaded, not the
# generator algorithm, seed, geometry, severity, or appearance.
# ================================================================

import textwrap
import subprocess
import time
import json
import hashlib
from pathlib import Path

# ----------------------------------------------------------------
# Locate the ISIC image directory
# ----------------------------------------------------------------

IMAGE_DIR = (
    globals().get("IMAGE_DIR_OVERRIDE")
    or find_image_dir(INPUT, min_images=100)
)

assert IMAGE_DIR is not None, (
    "ISIC image folder not found anywhere under /kaggle/input. "
    "Check that the isic-2019-dataset-full dataset is attached "
    "through Add Data."
)

print("image folder:", IMAGE_DIR)


# ----------------------------------------------------------------
# Required project paths
# ----------------------------------------------------------------

GEN_PATH = (
    PROJECT
    / "step1J_generator_recovery_lock"
    / "recovered_original_hair_generator.py"
)

POP_PATH = (
    PROJECT
    / "step0I_locked_clean_population"
    / "LOCKED_CLEAN_development_population.csv"
)

CACHE_LOCK = RECORDS / "cache_final_hash.json"

CACHE.mkdir(
    parents=True,
    exist_ok=True
)

assert GEN_PATH.exists(), (
    f"Recovered hair generator not found:\n{GEN_PATH}"
)

assert POP_PATH.exists(), (
    f"Locked development population not found:\n{POP_PATH}"
)


# ----------------------------------------------------------------
# Check whether a COMPLETE cache already exists
# ----------------------------------------------------------------

REQUIRED_CACHE_FILES = [
    "clean_224.npy",
    "hair_224.npy",
    "mask_area.npy",
    "cache_index.csv",
]

cache_files_present = all(
    (CACHE / f).exists()
    for f in REQUIRED_CACHE_FILES
)

if (
    CACHE_LOCK.exists()
    and cache_files_present
    and not globals().get("ALLOW_RERUN_CACHE", False)
):

    print("=" * 80)
    print("STAGE 2 — EXISTING CACHE DETECTED")
    print("=" * 80)

    print(
        "✓ cache_final_hash.json exists"
    )

    for f in REQUIRED_CACHE_FILES:
        print(
            "✓",
            CACHE / f
        )

    print(
        "\n✓ Complete cache already exists."
    )

    print(
        "Skipping rebuild."
    )

else:

    # ============================================================
    # Remove any PARTIAL cache from a previous failed attempt
    # ============================================================

    print("=" * 80)
    print("STAGE 2 — PREPARING CLEAN CACHE BUILD")
    print("=" * 80)

    if CACHE_LOCK.exists():
        print(
            "Removing stale/incomplete cache lock:",
            CACHE_LOCK
        )
        CACHE_LOCK.unlink()

    for name in [
        "clean_224.npy",
        "hair_224.npy",
        "mask_area.npy",
        "cache_index.csv",
        "_builder_record.json",
    ]:

        p = CACHE / name

        if p.exists():
            p.unlink()
            print(
                "removed:",
                p
            )

    print(
        "\n✓ Partial cache cleared."
    )


    # ============================================================
    # Write a standalone cache-builder subprocess
    # ============================================================

    BUILDER = TEMP / "cache_builder.py"

    BUILDER.write_text(
        textwrap.dedent(
            r'''
            import argparse
            import ast
            import hashlib
            import json
            import random
            from pathlib import Path

            import numpy as np
            import pandas as pd
            from PIL import Image
            import multiprocessing as mp


            # ====================================================
            # Fixed generator severity
            # ====================================================

            SEV = "medium"


            # ====================================================
            # SHA256 helper
            # ====================================================

            def sha_file(
                p,
                chunk=1 << 22
            ):

                h = hashlib.sha256()

                with open(p, "rb") as f:

                    for b in iter(
                        lambda: f.read(chunk),
                        b""
                    ):
                        h.update(b)

                return h.hexdigest()


            # ====================================================
            # Locate source image
            # ====================================================

            def find_image(
                image_dir,
                image_id
            ):

                for ext in (
                    ".jpg",
                    ".jpeg",
                    ".JPG",
                    ".JPEG"
                ):

                    p = (
                        image_dir
                        / f"{image_id}{ext}"
                    )

                    if p.exists():
                        return p

                raise FileNotFoundError(
                    image_id
                )


            # ====================================================
            # Global worker state
            # ====================================================

            _G = {}


            # ====================================================
            # SAFE GENERATOR LOADER
            #
            # The recovered generator file contains both the
            # reusable functions AND the original Step 1A test.
            #
            # We must NOT execute the whole file.
            #
            # We keep:
            #   - imports
            #   - literal constant assignments
            #   - function definitions
            #
            # We deliberately skip:
            #   - pd.read_csv(...)
            #   - loops
            #   - assertions
            #   - test execution
            #   - plotting
            #   - filesystem side effects
            # ====================================================

            def load_generator(
                gen_path
            ):

                source = Path(
                    gen_path
                ).read_text(
                    encoding="utf-8"
                )

                tree = ast.parse(
                    source
                )

                safe_nodes = []

                for node in tree.body:

                    # ------------------------------------------------
                    # Imports are required by generator functions
                    # ------------------------------------------------

                    if isinstance(
                        node,
                        (
                            ast.Import,
                            ast.ImportFrom,
                        )
                    ):

                        safe_nodes.append(
                            node
                        )

                        continue


                    # ------------------------------------------------
                    # Function definitions
                    # ------------------------------------------------

                    if isinstance(
                        node,
                        (
                            ast.FunctionDef,
                            ast.AsyncFunctionDef,
                        )
                    ):

                        safe_nodes.append(
                            node
                        )

                        continue


                    # ------------------------------------------------
                    # Only retain literal constant assignments.
                    #
                    # This captures things such as:
                    #
                    # GLOBAL_SEED = 20260919
                    # PRIMARY_SEVERITY = "medium"
                    # WIDTH_RANGE = {...}
                    #
                    # but NOT:
                    #
                    # manifest = pd.read_csv(...)
                    # sample_ids = ...
                    # IMAGE_DIR = Path(...)
                    # results = []
                    # etc.
                    # ------------------------------------------------

                    if isinstance(
                        node,
                        ast.Assign
                    ):

                        value = node.value

                        try:

                            ast.literal_eval(
                                value
                            )

                            safe_nodes.append(
                                node
                            )

                        except Exception:

                            pass


                    elif isinstance(
                        node,
                        ast.AnnAssign
                    ):

                        if node.value is not None:

                            try:

                                ast.literal_eval(
                                    node.value
                                )

                                safe_nodes.append(
                                    node
                                )

                            except Exception:

                                pass


                module = ast.Module(
                    body=safe_nodes,
                    type_ignores=[]
                )

                ast.fix_missing_locations(
                    module
                )

                ns = {
                    "__file__": str(gen_path),
                    "__name__": "recovered_hair_generator",
                }

                exec(
                    compile(
                        module,
                        str(gen_path),
                        "exec"
                    ),
                    ns
                )

                # ------------------------------------------------
                # Required functions MUST exist
                # ------------------------------------------------

                assert (
                    "generate_hair_mask"
                    in ns
                ), (
                    "generate_hair_mask was not loaded "
                    "from recovered generator"
                )

                assert (
                    "apply_hair"
                    in ns
                ), (
                    "apply_hair was not loaded "
                    "from recovered generator"
                )

                return ns


            # ====================================================
            # Worker initializer
            # ====================================================

            def _init(
                gen_path,
                cp,
                hp,
                image_dir
            ):

                ns = load_generator(
                    gen_path
                )

                _G.update(
                    gm=ns[
                        "generate_hair_mask"
                    ],

                    ah=ns[
                        "apply_hair"
                    ],

                    clean=np.load(
                        cp,
                        mmap_mode="r+"
                    ),

                    hair=np.load(
                        hp,
                        mmap_mode="r+"
                    ),

                    image_dir=Path(
                        image_dir
                    )
                )


            # ====================================================
            # Worker
            # ====================================================

            def _work(
                job
            ):

                lo, ids = job

                out = []

                for r, image_id in zip(
                    range(
                        lo,
                        lo + len(ids)
                    ),
                    ids
                ):

                    # --------------------------------------------
                    # Canonical 224x224 clean image
                    # --------------------------------------------

                    clean = np.asarray(
                        Image.open(
                            find_image(
                                _G["image_dir"],
                                image_id
                            )
                        )
                        .convert("RGB")
                        .resize(
                            (
                                224,
                                224
                            ),
                            Image.Resampling.LANCZOS
                        ),
                        dtype=np.uint8
                    )


                    # --------------------------------------------
                    # Deterministic hair mask
                    # --------------------------------------------

                    mask = _G["gm"](
                        image_id=image_id,
                        severity=SEV
                    )


                    # --------------------------------------------
                    # Deterministic hair image
                    # --------------------------------------------

                    hair = _G["ah"](
                        image=clean,
                        mask=mask,
                        severity=SEV,
                        image_id=image_id
                    )


                    # --------------------------------------------
                    # Causal-integrity check
                    #
                    # Pixels outside mask MUST remain unchanged.
                    # --------------------------------------------

                    m = mask > 0

                    outside = int(
                        (
                            np.any(
                                hair != clean,
                                axis=2
                            )
                            & ~m
                        ).sum()
                    )


                    # --------------------------------------------
                    # Write rows to memory-mapped arrays
                    # --------------------------------------------

                    _G["clean"][r] = clean

                    _G["hair"][r] = hair


                    out.append(
                        (
                            r,
                            int(m.sum()),
                            outside
                        )
                    )


                _G["clean"].flush()

                _G["hair"].flush()

                return out


            # ====================================================
            # MAIN
            # ====================================================

            def main():

                ap = argparse.ArgumentParser()

                ap.add_argument(
                    "--gen-path",
                    required=True
                )

                ap.add_argument(
                    "--pop-path",
                    required=True
                )

                ap.add_argument(
                    "--cache-dir",
                    required=True
                )

                ap.add_argument(
                    "--image-dir",
                    required=True
                )

                ap.add_argument(
                    "--n-proc",
                    type=int,
                    default=2
                )

                a = ap.parse_args()


                gen_path = Path(
                    a.gen_path
                )

                pop_path = Path(
                    a.pop_path
                )

                cache_dir = Path(
                    a.cache_dir
                )

                image_dir = Path(
                    a.image_dir
                )


                # =================================================
                # Load locked population
                # =================================================

                pop = pd.read_csv(
                    pop_path
                )

                assert list(
                    pop.columns
                ) == [
                    "image",
                    "lesion_id",
                    "class_name",
                    "malignant",
                    "split"
                ]


                # IMPORTANT:
                # Preserve the canonical ordering used by the
                # original cache construction.
                pop = (
                    pop
                    .sort_values(
                        [
                            "split",
                            "image"
                        ]
                    )
                    .reset_index(
                        drop=True
                    )
                )


                N = len(
                    pop
                )


                print(
                    f"locked population: {N} images",
                    flush=True
                )


                # =================================================
                # Cache index
                # =================================================

                idx = pop.copy()

                idx["row"] = np.arange(
                    N
                )

                idx.to_csv(
                    cache_dir
                    / "cache_index.csv",
                    index=False
                )


                # =================================================
                # Allocate memory-mapped arrays
                # =================================================

                clean_p = (
                    cache_dir
                    / "clean_224.npy"
                )

                hair_p = (
                    cache_dir
                    / "hair_224.npy"
                )

                area_p = (
                    cache_dir
                    / "mask_area.npy"
                )


                for p in (
                    clean_p,
                    hair_p
                ):

                    np.lib.format.open_memmap(
                        p,
                        mode="w+",
                        dtype=np.uint8,
                        shape=(
                            N,
                            224,
                            224,
                            3
                        )
                    ).flush()


                # =================================================
                # Jobs
                # =================================================

                ids = (
                    pop["image"]
                    .astype(str)
                    .tolist()
                )


                jobs = [
                    (
                        lo,
                        ids[
                            lo:
                            lo + 64
                        ]
                    )
                    for lo in range(
                        0,
                        N,
                        64
                    )
                ]


                area = np.zeros(
                    N,
                    dtype=np.int32
                )

                outside_total = 0

                done = 0


                # =================================================
                # Multiprocessing
                #
                # This subprocess has never imported CUDA.
                # Therefore fork is safe here.
                # =================================================

                print(
                    f"starting {a.n_proc} cache workers...",
                    flush=True
                )


                ctx = mp.get_context(
                    "fork"
                )


                with ctx.Pool(
                    a.n_proc,
                    initializer=_init,
                    initargs=(
                        str(gen_path),
                        str(clean_p),
                        str(hair_p),
                        str(image_dir),
                    )
                ) as pool:

                    for res in pool.imap_unordered(
                        _work,
                        jobs
                    ):

                        for (
                            r,
                            ar,
                            o
                        ) in res:

                            area[r] = ar

                            outside_total += o


                        done += len(
                            res
                        )


                        print(
                            f"cache {done:6d}/{N}",
                            flush=True
                        )


                # =================================================
                # Exact causal-integrity requirement
                # =================================================

                if outside_total != 0:

                    print(
                        f"FATAL: {outside_total} pixels changed "
                        f"outside the hair mask",
                        flush=True
                    )

                    raise SystemExit(
                        1
                    )


                # =================================================
                # Save mask areas
                # =================================================

                np.save(
                    area_p,
                    area
                )


                # =================================================
                # Deterministic spot checks
                # =================================================

                print(
                    "running deterministic 60-image spot check...",
                    flush=True
                )


                ns = load_generator(
                    gen_path
                )


                cm = np.load(
                    clean_p,
                    mmap_mode="r"
                )

                hm = np.load(
                    hair_p,
                    mmap_mode="r"
                )


                sample_rows = random.Random(
                    20260919
                ).sample(
                    range(N),
                    min(
                        60,
                        N
                    )
                )


                for r in sample_rows:

                    image_id = ids[r]


                    c = np.asarray(
                        Image.open(
                            find_image(
                                image_dir,
                                image_id
                            )
                        )
                        .convert("RGB")
                        .resize(
                            (
                                224,
                                224
                            ),
                            Image.Resampling.LANCZOS
                        ),
                        dtype=np.uint8
                    )


                    h = ns[
                        "apply_hair"
                    ](
                        image=c,
                        mask=ns[
                            "generate_hair_mask"
                        ](
                            image_id=image_id,
                            severity=SEV
                        ),
                        severity=SEV,
                        image_id=image_id
                    )


                    assert np.array_equal(
                        cm[r],
                        c
                    ), (
                        f"clean cache mismatch "
                        f"at row {r}"
                    )


                    assert np.array_equal(
                        hm[r],
                        h
                    ), (
                        f"hair cache mismatch "
                        f"at row {r}"
                    )


                print(
                    "SPOT_CHECK_OK",
                    flush=True
                )


                # =================================================
                # SHA256 records
                # =================================================

                file_sha = {
                    p.name: sha_file(p)
                    for p in (
                        clean_p,
                        hair_p,
                        area_p,
                        cache_dir
                        / "cache_index.csv"
                    )
                }


                record = {

                    "n_images": N,

                    "mask_area": {
                        "min": int(
                            area.min()
                        ),

                        "mean": float(
                            area.mean()
                        ),

                        "max": int(
                            area.max()
                        ),
                    },

                    "file_sha256": file_sha,

                    "generator_sha256": sha_file(
                        gen_path
                    ),

                    "image_dir": str(
                        image_dir
                    ),

                    "builder_protocol": (
                        "KAGGLE_SAFE_GENERATOR_IMPORT_V1"
                    ),

                    "generator_execution": (
                        "function_definitions_only"
                    ),
                }


                with open(
                    cache_dir
                    / "_builder_record.json",
                    "w"
                ) as f:

                    json.dump(
                        record,
                        f,
                        indent=2
                    )


                print(
                    "DONE",
                    flush=True
                )


            # ====================================================
            # Entry point
            # ====================================================

            if __name__ == "__main__":

                main()
            '''
        ),
        encoding="utf-8"
    )


    # ============================================================
    # Run builder as a completely fresh subprocess
    # ============================================================

    N_PROC = int(
        globals().get(
            "N_PROC_OVERRIDE",
            2
        )
    )


    assert N_PROC >= 1

    print(
        "\n" + "=" * 80
    )

    print(
        "STAGE 2 — STARTING CACHE BUILDER"
    )

    print(
        "=" * 80
    )

    print(
        "workers:",
        N_PROC
    )

    print(
        "generator:",
        GEN_PATH
    )

    print(
        "population:",
        POP_PATH
    )

    print(
        "cache:",
        CACHE
    )

    print(
        "=" * 80
    )


    t0 = time.time()

    proc = subprocess.Popen(

        [
            "python3",
            "-u",
            str(BUILDER),

            "--gen-path",
            str(GEN_PATH),

            "--pop-path",
            str(POP_PATH),

            "--cache-dir",
            str(CACHE),

            "--image-dir",
            str(IMAGE_DIR),

            "--n-proc",
            str(N_PROC),
        ],

        stdout=subprocess.PIPE,

        stderr=subprocess.STDOUT,

        text=True,

        bufsize=1
    )


    tail = []


    for line in proc.stdout:

        line = line.rstrip()

        elapsed = (
            time.time()
            - t0
        )

        print(
            f"  [{elapsed:7.0f}s] {line}",
            flush=True
        )

        tail.append(
            line
        )


    rc = proc.wait()


    # ============================================================
    # Builder failure
    # ============================================================

    if rc != 0:

        raise RuntimeError(
            "CACHE BUILDER FAILED "
            f"(return code {rc}). "
            "See the subprocess output above."
        )


    # ============================================================
    # Required spot-check result
    # ============================================================

    assert any(
        "SPOT_CHECK_OK" in line
        for line in tail
    ), (
        "Cache builder finished without "
        "a successful SPOT_CHECK_OK."
    )


    elapsed = (
        time.time()
        - t0
    )


    print(
        "\n✓ 60-image deterministic spot check passed."
    )

    print(
        f"✓ Cache build completed in {elapsed:.0f} seconds."
    )


    # ============================================================
    # Read builder record
    # ============================================================

    builder_record_path = (
        CACHE
        / "_builder_record.json"
    )


    assert builder_record_path.exists(), (
        "Builder finished but _builder_record.json "
        "was not created."
    )


    with open(
        builder_record_path,
        "r"
    ) as f:

        rec = json.load(
            f
        )


    # ============================================================
    # Verify expected cache files
    # ============================================================

    for f in REQUIRED_CACHE_FILES:

        assert (
            CACHE / f
        ).exists(), (
            f"Missing cache file: {CACHE / f}"
        )


    # ============================================================
    # Save canonical cache records
    # ============================================================

    save_record(
        "cache_record.json",
        rec
    )


    cache_hash = hashlib.sha256(
        json.dumps(
            rec["file_sha256"],
            sort_keys=True
        ).encode()
    ).hexdigest()


    save_record(
        "cache_final_hash.json",
        {
            "sha256": cache_hash,
            "status": "LOCKED",
        }
    )


    # ============================================================
    # Final report
    # ============================================================

    print(
        "\n" + "=" * 80
    )

    print(
        "STAGE 2 — CACHE BUILT SUCCESSFULLY"
    )

    print(
        "=" * 80
    )

    print(
        "images:",
        rec["n_images"]
    )

    print(
        "mask area min:",
        rec["mask_area"]["min"]
    )

    print(
        "mask area mean:",
        f'{rec["mask_area"]["mean"]:.2f}'
    )

    print(
        "mask area max:",
        rec["mask_area"]["max"]
    )

    print(
        "\nCache SHA256:"
    )

    print(
        cache_hash
    )

    print(
        "\nFiles:"
    )

    for f in REQUIRED_CACHE_FILES:

        print(
            "✓",
            CACHE / f
        )

    print(
        "\nSTATUS: COMPLETE"
    )

    print(
        "=" * 80
    )

image folder: /kaggle/input/datasets/alifshahariar/isic-2019-dataset-full/isic-2019-dataset-full/train
STAGE 2 — PREPARING CLEAN CACHE BUILD

✓ Partial cache cleared.

STAGE 2 — STARTING CACHE BUILDER
workers: 4
generator: /kaggle/working/isic_final_project/step1J_generator_recovery_lock/recovered_original_hair_generator.py
population: /kaggle/working/isic_final_project/step0I_locked_clean_population/LOCKED_CLEAN_development_population.csv
cache: /kaggle/temp/isic_cache
  [      1s] locked population: 23227 images
  [      1s] starting 4 cache workers...
  [      6s] cache     64/23227
  [      6s] cache    128/23227
  [      6s] cache    192/23227
  [      6s] cache    256/23227
  [     10s] cache    320/23227
  [     10s] cache    384/23227
  [     10s] cache    448/23227
  [     10s] cache    512/23227
  [     14s] cache    576/23227
  [     14s] cache    640/23227
  [     14s] cache    704/23227
  [     15s] cache    768/23227
  [     18s] cache    832/23227
  [     18s] cache    8

### Stage 1b — pin package versions
Matches the training environment's torch/torchvision for consistency. Restart the kernel if this errors about torch already being imported (same rule as the training notebook).

In [7]:
# ================================================================
# STAGE 1 — PIN PACKAGE VERSIONS  (analysis uses the same torch/torchvision as training, for consistency - not
# strictly required for correctness the way training determinism was, but avoids any subtle behavioral difference)
#
# IMPORTANT: this cell must be the FIRST thing in this kernel session to touch torch/torchvision. If you already ran
# it once (or ran any cell that imports torch) in this session, RESTART THE KERNEL and run the notebook again from
# Cell 1 - a pip-installed new version cannot replace an already-imported module in a live Python process.
# ================================================================
import subprocess, importlib.metadata as ilmd

if "torch" in sys.modules or "torchvision" in sys.modules:
    raise RuntimeError(
        "torch/torchvision are already imported in this kernel session, so pinning them now would silently do nothing "
        "(pip can replace the files on disk, but not an already-loaded module in memory). "
        "RESTART THE KERNEL (Kernel -> Restart & Clear Output) and re-run the notebook from Cell 1, without running this "
        "stage more than once per session.")

LOCKED_VERSIONS = {"torch": "2.11.0", "torchvision": "0.26.0"}   # numpy/Pillow: locked project did not pin exact versions
CUDA_TAG = "cu128"
PYTORCH_INDEX = f"https://download.pytorch.org/whl/{CUDA_TAG}"

def dist_version(pkg):
    # reads installed package METADATA only - never imports the package, so it can't poison sys.modules
    try:
        return ilmd.version(pkg)
    except ilmd.PackageNotFoundError:
        return None

before = {p: dist_version(p) for p in LOCKED_VERSIONS}
print("Kaggle default versions (from package metadata, not imported):", before)

pin_result = {"attempted": False, "returncode": None, "stdout_tail": None}
if AUTO_PIN_VERSIONS:
    already_ok = all((before[p] or "").split("+")[0] == v for p, v in LOCKED_VERSIONS.items())
    if already_ok:
        print("✓ torch/torchvision already match the locked versions - nothing to install")
    else:
        pin_result["attempted"] = True
        pkgs = [f"{p}=={v}+{CUDA_TAG}" for p, v in LOCKED_VERSIONS.items()]
        log(f"pinning to {pkgs} from {PYTORCH_INDEX} (installs only; does not import) ...")
        r = subprocess.run(["pip", "install", "-q", "--index-url", PYTORCH_INDEX] + pkgs, capture_output=True, text=True)
        pin_result["returncode"] = r.returncode
        pin_result["stdout_tail"] = (r.stdout[-1500:] + r.stderr[-1500:])
        if r.returncode != 0:
            print(pin_result["stdout_tail"])
            log("exact +cu128 pin failed (see above) - retrying with a plain version pin from the same CUDA index ...")
            r2 = subprocess.run(["pip", "install", "-q", "--index-url", PYTORCH_INDEX] +
                                [f"{p}=={v}" for p, v in LOCKED_VERSIONS.items()], capture_output=True, text=True)
            pin_result["returncode"] = r2.returncode
            pin_result["stdout_tail"] = (r2.stdout[-1500:] + r2.stderr[-1500:])
            if r2.returncode != 0:
                print(pin_result["stdout_tail"])
        after = {p: dist_version(p) for p in LOCKED_VERSIONS}
        print("Versions after pinning attempt:", after)


# ---- first import of torch in this session happens here, AFTER any pinning, so it loads the pinned files fresh
import torch, torchvision, numpy, PIL
env = {
    "torch_version": torch.__version__, "torchvision_version": torchvision.__version__,
    "cuda_version": torch.version.cuda, "cudnn_version": torch.backends.cudnn.version(),
    "gpu_count": torch.cuda.device_count(), "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
}
env["matches_locked_torch_torchvision"] = (torch.__version__.split("+")[0] == LOCKED_VERSIONS["torch"]
                                           and torchvision.__version__.split("+")[0] == LOCKED_VERSIONS["torchvision"])
assert torch.cuda.is_available(), "No GPU visible. Set Settings -> Accelerator -> GPU T4 x2, then Save and re-run from Cell 1."
print("Analysis environment on Kaggle:", json.dumps(env, indent=2))
if not env["matches_locked_torch_torchvision"]:
    print(f"\ntorch/torchvision do not match the training versions {LOCKED_VERSIONS}. This affects only consistency, not "
          "correctness of the analysis - the frozen analysis_pipeline.py and each model's own trained weights are unaffected.")
    assert ALLOW_VERSION_DEVIATION, "Set ALLOW_VERSION_DEVIATION = True to proceed anyway, or investigate the pinning output above."
save_record("analysis_environment.json", env)
print("\n✓ STAGE 1 done.")


Kaggle default versions (from package metadata, not imported): {'torch': '2.10.0+cu128', 'torchvision': '0.25.0+cu128'}
[ 0.18 h] pinning to ['torch==2.11.0+cu128', 'torchvision==0.26.0+cu128'] from https://download.pytorch.org/whl/cu128 (installs only; does not import) ...
Versions after pinning attempt: {'torch': '2.11.0+cu128', 'torchvision': '0.26.0+cu128'}
Analysis environment on Kaggle: {
  "torch_version": "2.11.0+cu128",
  "torchvision_version": "0.26.0+cu128",
  "cuda_version": "12.8",
  "cudnn_version": 91900,
  "gpu_count": 2,
  "gpus": [
    "Tesla T4",
    "Tesla T4"
  ],
  "matches_locked_torch_torchvision": true
}

✓ STAGE 1 done.


### Stage 2 — extract logits for all 30 models
For each model, both checkpoints (final-epoch-15 = primary, best-eval-BA = sensitivity), computes z = z_M - z_B under clean/hair/reversed/matched/neutral. One .npz per (model, checkpoint) - resumable, skips what's already done.

In [8]:
# ================================================================
# STAGE 2 — EXTRACT LOGITS FOR ALL 30 MODELS
# (final-epoch-15 checkpoint, primary; best-eval-BA checkpoint, sensitivity)
#
# For each model, computes z = z_M - z_B on:
#   clean, hair, reversed, matched, neutral
#
# One .npz per (model, checkpoint) with the per-image z arrays.
# This is data extraction, not endpoint computation.
# ================================================================

import importlib.util
import numpy as np
import pandas as pd
import torch
from torchvision import models
import torch.nn as nn

# ----------------------------------------------------------------
# Load frozen analysis pipeline
# ----------------------------------------------------------------

spec = importlib.util.spec_from_file_location(
    "analysis_pipeline",
    ANALYSIS_MODULE
)

ap = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ap)

device = torch.device("cuda")

# ----------------------------------------------------------------
# Output directory
# ----------------------------------------------------------------

LOGIT_DIR = RESULTS / "logits"
LOGIT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------
# Load cache index and fixed evaluation population
# ----------------------------------------------------------------

idx = pd.read_csv(
    CACHE / "cache_index.csv"
)

ev = (
    idx[idx["split"] == "val"]
    .sort_values("image")
    .reset_index(drop=True)
)

ev_rows = ev["row"].to_numpy()
ev_y = ev["malignant"].to_numpy()

# Saved once for downstream Stage 3
np.save(
    RESULTS / "eval_y.npy",
    ev_y
)

# ----------------------------------------------------------------
# Memory-map clean and hair caches
# ----------------------------------------------------------------

clean_arr = np.load(
    CACHE / "clean_224.npy",
    mmap_mode="r"
)

hair_arr = np.load(
    CACHE / "hair_224.npy",
    mmap_mode="r"
)

# ----------------------------------------------------------------
# Load locked evaluation manifests
# ----------------------------------------------------------------

manifests = {}

for cond in ("reversed", "matched", "neutral"):

    m = (
        pd.read_csv(
            PROJECT
            / "step2R_production_evaluation_manifests"
            / f"evaluation_{cond}.csv"
        )
        .sort_values("image")
        .reset_index(drop=True)
    )

    assert (
        m["image"].to_numpy()
        == ev["image"].to_numpy()
    ).all(), (
        f"{cond} manifest image order "
        "does not match the cache index"
    )

    manifests[cond] = (
        m["hair_present"]
        .to_numpy()
        .astype(bool)
    )

# ----------------------------------------------------------------
# Hair-presence flags for every evaluation condition
# ----------------------------------------------------------------

hair_flags_all = {
    "hair": np.ones(
        len(ev),
        dtype=bool
    ),

    "clean": np.zeros(
        len(ev),
        dtype=bool
    ),

    "reversed": manifests["reversed"],

    "matched": manifests["matched"],

    "neutral": manifests["neutral"],
}

# ----------------------------------------------------------------
# Model construction
# ----------------------------------------------------------------

def build_model(gap):

    m = models.resnet18(
        weights=None
    )

    m.fc = nn.Linear(
        m.fc.in_features,
        2
    )

    if gap == "mean":

        class MeanPool(nn.Module):

            def forward(self, x):
                return x.mean(
                    dim=(2, 3),
                    keepdim=True
                )

        m.avgpool = MeanPool()

    return m.to(device).eval()


# ----------------------------------------------------------------
# Locked run order
# ----------------------------------------------------------------

RUN_ORDER = json.load(
    open(
        PROJECT
        / "step2V_seed_table_and_run_order"
        / "RUN_ORDER_MANIFEST.json"
    )
)["run_order"]

all_names = [
    f"rho_{o['rho']:.1f}_rep_{o['replicate']}"
    for o in RUN_ORDER
]

# ----------------------------------------------------------------
# Output filename helpers
# ----------------------------------------------------------------

def npz_path(name, ckpt_label):
    return (
        LOGIT_DIR
        / f"{name}__{ckpt_label}.npz"
    )


def already_done(name, ckpt_label):

    p = npz_path(
        name,
        ckpt_label
    )

    if not p.exists():
        return False

    try:
        with np.load(p) as data:
            return set(data.files) == {
                "z_clean",
                "z_hair",
                "z_reversed",
                "z_matched",
                "z_neutral",
            }

    except Exception:
        # Treat a corrupt/incomplete NPZ as not completed.
        return False


# ----------------------------------------------------------------
# Determine models still requiring extraction
# ----------------------------------------------------------------

todo = [
    o
    for o in RUN_ORDER
    if not (
        already_done(
            f"rho_{o['rho']:.1f}_rep_{o['replicate']}",
            "final"
        )
        and
        already_done(
            f"rho_{o['rho']:.1f}_rep_{o['replicate']}",
            "best"
        )
    )
]

if MAX_MODELS_THIS_RUN is not None:
    todo = todo[:MAX_MODELS_THIS_RUN]

n_before = sum(
    1
    for n in all_names
    if (
        already_done(n, "final")
        and
        already_done(n, "best")
    )
)

print(
    f"{n_before}/30 models already extracted "
    f"(both checkpoints) - {len(todo)} to do this run."
)

# ----------------------------------------------------------------
# Build model once and reuse it
# ----------------------------------------------------------------

model = build_model(
    GAP_DECISION
)

# ----------------------------------------------------------------
# Extract logits
# ----------------------------------------------------------------

for o in todo:

    if (
        elapsed_h()
        + EST_MODEL_HOURS
        > DEADLINE_HOURS
    ):

        print(
            "Stopping before the deadline; "
            "commit now, resume next session."
        )

        break

    name = (
        f"rho_{o['rho']:.1f}_rep_{o['replicate']}"
    )

    ck_dir = TR / name

    # ------------------------------------------------------------
    # Process both locked checkpoints
    # ------------------------------------------------------------

    for ckpt_name, ckpt_label in (
        ("final_epoch15.pt", "final"),
        ("best_eval_ba.pt", "best"),
    ):

        if already_done(
            name,
            ckpt_label
        ):
            continue

        # --------------------------------------------------------
        # Load checkpoint
        # --------------------------------------------------------

        checkpoint_path = (
            ck_dir / ckpt_name
        )

        assert checkpoint_path.exists(), (
            f"Checkpoint not found:\n"
            f"{checkpoint_path}"
        )

        sd = torch.load(
            checkpoint_path,
            map_location=device
        )

        model.load_state_dict(
            sd
        )

        model.eval()

        # --------------------------------------------------------
        # Extract all five conditions
        # --------------------------------------------------------

        z = {}

        with torch.no_grad():

            for cond, flags in (
                hair_flags_all.items()
            ):

                z[f"z_{cond}"] = (
                    ap.extract_condition_logits(
                        model,
                        clean_arr,
                        hair_arr,
                        ev_rows,
                        flags,
                        device
                    )
                )

        # --------------------------------------------------------
        # Atomic NPZ save
        #
        # IMPORTANT:
        # np.savez("file.npz.tmp", ...)
        # would create:
        #     file.npz.tmp.npz
        #
        # Opening the temporary file explicitly prevents NumPy
        # from appending ".npz".
        # --------------------------------------------------------

        p = npz_path(
            name,
            ckpt_label
        )

        tmp = p.with_suffix(
            ".npz.tmp"
        )

        with open(
            tmp,
            "wb"
        ) as f:

            np.savez(
                f,
                **z
            )

        # Replace destination only after successful write
        os.replace(
            tmp,
            p
        )

    log(
        f"✓ {name} extracted "
        f"(both checkpoints)"
    )

# ----------------------------------------------------------------
# Final completion check
# ----------------------------------------------------------------

n_after = sum(
    1
    for n in all_names
    if (
        already_done(n, "final")
        and
        already_done(n, "best")
    )
)

print(
    f"\nSTAGE 2: {n_after}/30 models extracted."
)

if n_after == 30:

    print(
        "All models extracted - "
        "continue to Stage 3 "
        "(endpoints, IG, statistics)."
    )

else:

    print(
        "Not all models done - "
        "commit this version, then resume "
        "(Add Data -> Your Work -> this notebook) "
        "next session."
    )

0/30 models already extracted (both checkpoints) - 30 to do this run.
[ 0.23 h] ✓ rho_0.6_rep_2 extracted (both checkpoints)
[ 0.25 h] ✓ rho_0.8_rep_3 extracted (both checkpoints)
[ 0.27 h] ✓ rho_1.0_rep_5 extracted (both checkpoints)
[ 0.29 h] ✓ rho_0.0_rep_5 extracted (both checkpoints)
[ 0.30 h] ✓ rho_1.0_rep_3 extracted (both checkpoints)
[ 0.32 h] ✓ rho_0.6_rep_4 extracted (both checkpoints)
[ 0.34 h] ✓ rho_0.8_rep_2 extracted (both checkpoints)
[ 0.36 h] ✓ rho_0.2_rep_4 extracted (both checkpoints)
[ 0.38 h] ✓ rho_0.2_rep_1 extracted (both checkpoints)
[ 0.40 h] ✓ rho_0.4_rep_3 extracted (both checkpoints)
[ 0.42 h] ✓ rho_0.4_rep_1 extracted (both checkpoints)
[ 0.44 h] ✓ rho_0.0_rep_4 extracted (both checkpoints)
[ 0.46 h] ✓ rho_0.0_rep_2 extracted (both checkpoints)
[ 0.48 h] ✓ rho_1.0_rep_4 extracted (both checkpoints)
[ 0.49 h] ✓ rho_0.6_rep_1 extracted (both checkpoints)
[ 0.51 h] ✓ rho_0.0_rep_1 extracted (both checkpoints)
[ 0.53 h] ✓ rho_0.0_rep_3 extracted (both checkpoi

### Stage 3 — Integrated Gradients on the 500-image XAI subset
Final-epoch-15 checkpoint only, as pre-declared. Slower per model than Stage 2 - the deadline check here is 3x more conservative for that reason. One .npz per model with per-image enrichment and completeness diagnostics.

In [9]:
# ================================================================
# STAGE 3 — INTEGRATED GRADIENTS on the locked 500-image XAI subset
#
# Final-epoch-15 checkpoint only, as pre-declared.
#
# One .npz per model with:
#   - per-image enrichment
#   - completeness error
#   - undefined flag
#   - steps used
#   - signed attribution inside mask
#
# Resumable like Stage 2.
# ================================================================

import ast
import hashlib
import json
import numpy as np
import pandas as pd
import torch
from torchvision import models
import torch.nn as nn

device = torch.device("cuda")

# ----------------------------------------------------------------
# Load frozen analysis pipeline
# ----------------------------------------------------------------

spec = importlib.util.spec_from_file_location(
    "analysis_pipeline",
    ANALYSIS_MODULE
)

ap = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ap)

# ----------------------------------------------------------------
# Output directory
# ----------------------------------------------------------------

IG_DIR = RESULTS / "ig"
IG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ----------------------------------------------------------------
# Locked XAI subset
# ----------------------------------------------------------------

xai = pd.read_csv(
    PROJECT
    / "step2U_xai_subset_lock"
    / "LOCKED_XAI_SUBSET.csv"
)

assert len(xai) == 500, (
    f"Expected locked XAI subset of 500 images, "
    f"found {len(xai)}"
)

idx = pd.read_csv(
    CACHE / "cache_index.csv"
)

row_of = dict(
    zip(
        idx["image"],
        idx["row"]
    )
)

assert xai["image"].isin(
    row_of
).all(), (
    "Some XAI subset images are missing "
    "from the cache index."
)

xai_rows = xai["image"].map(
    row_of
).to_numpy()

# ----------------------------------------------------------------
# Hair cache
# ----------------------------------------------------------------

hair_arr = np.load(
    CACHE / "hair_224.npy",
    mmap_mode="r"
)

# ----------------------------------------------------------------
# SAFE GENERATOR LOADER
#
# Same construction principle as the locked Training Stage 2:
# execute only imports, literal constants, and function definitions.
# Do NOT execute the generator's module-level validation/test code.
# ----------------------------------------------------------------

GEN_PATH = (
    PROJECT
    / "step1J_generator_recovery_lock"
    / "recovered_original_hair_generator.py"
)


def load_generator_safe(
    gen_path
):

    source = Path(
        gen_path
    ).read_text(
        encoding="utf-8"
    )

    tree = ast.parse(
        source
    )

    safe_nodes = []

    for node in tree.body:

        # --------------------------------------------------------
        # Imports
        # --------------------------------------------------------

        if isinstance(
            node,
            (
                ast.Import,
                ast.ImportFrom,
            )
        ):

            safe_nodes.append(
                node
            )

            continue

        # --------------------------------------------------------
        # Function definitions
        # --------------------------------------------------------

        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            )
        ):

            safe_nodes.append(
                node
            )

            continue

        # --------------------------------------------------------
        # Literal-only constants
        # --------------------------------------------------------

        if isinstance(
            node,
            ast.Assign
        ):

            try:

                ast.literal_eval(
                    node.value
                )

                safe_nodes.append(
                    node
                )

            except Exception:

                pass

        elif isinstance(
            node,
            ast.AnnAssign
        ):

            if node.value is not None:

                try:

                    ast.literal_eval(
                        node.value
                    )

                    safe_nodes.append(
                        node
                    )

                except Exception:

                    pass

    module = ast.Module(
        body=safe_nodes,
        type_ignores=[]
    )

    ast.fix_missing_locations(
        module
    )

    ns = {
        "__file__": str(gen_path),
        "__name__": "recovered_hair_generator",
    }

    exec(
        compile(
            module,
            str(gen_path),
            "exec"
        ),
        ns
    )

    assert (
        "generate_hair_mask" in ns
    ), (
        "generate_hair_mask was not loaded "
        "from recovered generator"
    )

    assert (
        "apply_hair" in ns
    ), (
        "apply_hair was not loaded "
        "from recovered generator"
    )

    return ns


gen_ns = load_generator_safe(
    GEN_PATH
)

# ----------------------------------------------------------------
# Rebuild each XAI image's deterministic hair mask
#
# Same generator, same image IDs, same severity as the cache.
# ----------------------------------------------------------------

xai_masks = {}

for img in xai["image"]:

    xai_masks[img] = (
        gen_ns["generate_hair_mask"](
            image_id=img,
            severity="medium"
        ) > 0
    )

# ----------------------------------------------------------------
# Model construction
# ----------------------------------------------------------------

def build_model(gap):

    m = models.resnet18(
        weights=None
    )

    m.fc = nn.Linear(
        m.fc.in_features,
        2
    )

    if gap == "mean":

        class MeanPool(nn.Module):

            def forward(self, x):
                return x.mean(
                    dim=(2, 3),
                    keepdim=True
                )

        m.avgpool = MeanPool()

    return m.to(device).eval()


# ----------------------------------------------------------------
# Locked run order
# ----------------------------------------------------------------

RUN_ORDER = json.load(
    open(
        PROJECT
        / "step2V_seed_table_and_run_order"
        / "RUN_ORDER_MANIFEST.json"
    )
)["run_order"]

# ----------------------------------------------------------------
# Output helpers
# ----------------------------------------------------------------

def npz_path(name):

    return (
        IG_DIR
        / f"{name}.npz"
    )


def already_done(name):

    p = npz_path(
        name
    )

    if not p.exists():
        return False

    try:

        with np.load(p) as data:

            required = {
                "image",
                "enrichment",
                "completeness_error",
                "undefined",
                "steps_used",
                "signed_attribution",
            }

            return required.issubset(
                set(data.files)
            )

    except Exception:

        return False


# ----------------------------------------------------------------
# Determine models still requiring IG
# ----------------------------------------------------------------

todo = [
    o
    for o in RUN_ORDER
    if not already_done(
        f"rho_{o['rho']:.1f}_rep_{o['replicate']}"
    )
]

if MAX_MODELS_THIS_RUN is not None:

    todo = todo[
        :MAX_MODELS_THIS_RUN
    ]

n_before = sum(
    1
    for o in RUN_ORDER
    if already_done(
        f"rho_{o['rho']:.1f}_rep_{o['replicate']}"
    )
)

print(
    f"{n_before}/30 models already have IG results "
    f"- {len(todo)} to do this run."
)

# ----------------------------------------------------------------
# Build model once and reuse it
# ----------------------------------------------------------------

model = build_model(
    GAP_DECISION
)

# ----------------------------------------------------------------
# Integrated Gradients
# ----------------------------------------------------------------

for o in todo:

    if (
        elapsed_h()
        + EST_MODEL_HOURS * 3
        > DEADLINE_HOURS
    ):

        print(
            "Stopping before the deadline; "
            "commit now, resume next session."
        )

        break

    name = (
        f"rho_{o['rho']:.1f}_rep_{o['replicate']}"
    )

    checkpoint_path = (
        TR
        / name
        / "final_epoch15.pt"
    )

    assert checkpoint_path.exists(), (
        f"Primary checkpoint not found:\n"
        f"{checkpoint_path}"
    )

    # ------------------------------------------------------------
    # Load final epoch-15 checkpoint only
    # ------------------------------------------------------------

    sd = torch.load(
        checkpoint_path,
        map_location=device
    )

    model.load_state_dict(
        sd
    )

    model.eval()

    # ------------------------------------------------------------
    # Per-image IG results
    # ------------------------------------------------------------

    enrich = []
    err = []
    undef = []
    steps_used = []
    signed = []

    for img in xai["image"]:

        r = row_of[img]

        res = ap.ig_enrichment(
            model,
            np.array(
                hair_arr[r]
            ),
            xai_masks[img],
            device
        )

        enrich.append(
            res["enrichment"]
        )

        err.append(
            res["completeness_error"]
        )

        undef.append(
            res["undefined"]
        )

        steps_used.append(
            res["steps_used"]
        )

        signed.append(
            res["signed_attribution_inside_mask"]
        )

    # ------------------------------------------------------------
    # Atomic NPZ save
    #
    # IMPORTANT:
    # Do NOT call np.savez() directly on a filename ending in
    # ".npz.tmp", because NumPy will append another ".npz".
    # ------------------------------------------------------------

    p = npz_path(
        name
    )

    tmp = p.with_suffix(
        ".npz.tmp"
    )

    with open(
        tmp,
        "wb"
    ) as f:

        np.savez(
            f,
            image=xai["image"].to_numpy(),
            enrichment=np.array(
                enrich
            ),
            completeness_error=np.array(
                err
            ),
            undefined=np.array(
                undef
            ),
            steps_used=np.array(
                steps_used
            ),
            signed_attribution=np.array(
                signed
            )
        )

    os.replace(
        tmp,
        p
    )

    # ------------------------------------------------------------
    # Progress report
    # ------------------------------------------------------------

    frac_undef = np.mean(
        undef
    )

    defined_enrichment = np.where(
        np.asarray(undef, dtype=bool),
        np.nan,
        np.asarray(enrich, dtype=float)
    )

    log(
        f"✓ {name} IG done  |  "
        f"mean enrichment (defined only)="
        f"{np.nanmean(defined_enrichment):.2f}  "
        f"undefined_frac={frac_undef:.3f}"
    )

# ----------------------------------------------------------------
# Final completion check
# ----------------------------------------------------------------

n_after = sum(
    1
    for o in RUN_ORDER
    if already_done(
        f"rho_{o['rho']:.1f}_rep_{o['replicate']}"
    )
)

print(
    f"\nSTAGE 3: {n_after}/30 models have IG results."
)

if n_after == 30:

    print(
        "All models done - continue to Stage 4 "
        "(endpoints + confirmatory statistics)."
    )

else:

    print(
        "Not all models done - commit this version, "
        "then resume next session."
    )

0/30 models already have IG results - 30 to do this run.
[ 0.81 h] ✓ rho_0.6_rep_2 IG done  |  mean enrichment (defined only)=11.35  undefined_frac=0.000
[ 0.84 h] ✓ rho_0.8_rep_3 IG done  |  mean enrichment (defined only)=14.23  undefined_frac=0.000
[ 0.87 h] ✓ rho_1.0_rep_5 IG done  |  mean enrichment (defined only)=16.58  undefined_frac=0.000
[ 0.91 h] ✓ rho_0.0_rep_5 IG done  |  mean enrichment (defined only)=6.74  undefined_frac=0.000
[ 0.94 h] ✓ rho_1.0_rep_3 IG done  |  mean enrichment (defined only)=16.56  undefined_frac=0.000
[ 0.98 h] ✓ rho_0.6_rep_4 IG done  |  mean enrichment (defined only)=11.82  undefined_frac=0.000
[ 1.02 h] ✓ rho_0.8_rep_2 IG done  |  mean enrichment (defined only)=13.95  undefined_frac=0.000
[ 1.06 h] ✓ rho_0.2_rep_4 IG done  |  mean enrichment (defined only)=10.21  undefined_frac=0.000
[ 1.09 h] ✓ rho_0.2_rep_1 IG done  |  mean enrichment (defined only)=8.86  undefined_frac=0.000
[ 1.13 h] ✓ rho_0.4_rep_3 IG done  |  mean enrichment (defined only)=11.

### Stage 4 — endpoints + the confirmatory analysis
The ONLY place any endpoint or statistical test is computed, and only after asserting all 30 models have both Stage 2 and Stage 3 results. Runs H1, H2 (Holm-corrected), all pre-declared secondaries, the rho=1 vs rho=0 contrasts, Case F (churn reference) if H1 doesn't reject, and the best-checkpoint sensitivity check - all through the frozen `analysis_pipeline.py`, no new statistical code written here.

In [10]:
# ================================================================
# STAGE 4 — COMPUTE ENDPOINTS + RUN THE CONFIRMATORY ANALYSIS
#
# Only meaningful once Stages 2 and 3 both show 30/30.
#
# Uses ONLY the frozen analysis_pipeline.py (ap).
# No new statistical methodology is introduced here.
# ================================================================

import numpy as np
import pandas as pd
import json


# ----------------------------------------------------------------
# Paths / locked evaluation labels
# ----------------------------------------------------------------

ev_y = np.load(
    RESULTS / "eval_y.npy"
)

RUN_ORDER = json.load(
    open(
        PROJECT
        / "step2V_seed_table_and_run_order"
        / "RUN_ORDER_MANIFEST.json"
    )
)["run_order"]

LOGIT_DIR = RESULTS / "logits"
IG_DIR = RESULTS / "ig"


# ----------------------------------------------------------------
# Basic evaluation-label sanity check
# ----------------------------------------------------------------

assert len(ev_y) > 0, (
    "eval_y.npy is empty."
)

print(
    f"Evaluation images: {len(ev_y)}"
)


# ----------------------------------------------------------------
# HARD GATE
#
# Every model must have:
#   - final checkpoint logits
#   - best checkpoint logits
#   - IG results
# ----------------------------------------------------------------

for o in RUN_ORDER:

    name = (
        f"rho_{o['rho']:.1f}_rep_{o['replicate']}"
    )

    # ------------------------------------------------------------
    # Logit files
    # ------------------------------------------------------------

    for ck in (
        "final",
        "best"
    ):

        p = (
            LOGIT_DIR
            / f"{name}__{ck}.npz"
        )

        assert p.exists(), (
            f"Missing extraction for {name} ({ck}). "
            "Stage 2 is not complete."
        )

        with np.load(p) as z:

            required_logits = {
                "z_clean",
                "z_hair",
                "z_reversed",
                "z_matched",
                "z_neutral",
            }

            assert required_logits.issubset(
                set(z.files)
            ), (
                f"Incomplete logit file for "
                f"{name} ({ck})."
            )

            for key in required_logits:

                assert len(z[key]) == len(ev_y), (
                    f"Length mismatch in "
                    f"{name} ({ck}) -> {key}: "
                    f"{len(z[key])} vs {len(ev_y)}"
                )

    # ------------------------------------------------------------
    # IG result
    #
    # IMPORTANT:
    # Do NOT access ig["image"] here.
    # Stage 3 stored image IDs as an object array, which newer
    # NumPy refuses to load with allow_pickle=False.
    #
    # Stage 4 does not need the image IDs.
    # ------------------------------------------------------------

    ig_path = (
        IG_DIR
        / f"{name}.npz"
    )

    assert ig_path.exists(), (
        f"Missing IG results for {name}. "
        "Stage 3 is not complete."
    )

    with np.load(ig_path) as ig:

        required_ig = {
            "image",
            "enrichment",
            "completeness_error",
            "undefined",
            "steps_used",
            "signed_attribution",
        }

        # Check the field names without loading the object array.
        assert required_ig.issubset(
            set(ig.files)
        ), (
            f"Incomplete IG file for {name}."
        )

        # Use numeric enrichment array for the 500-image check.
        assert len(ig["enrichment"]) == 500, (
            f"{name}: expected 500 XAI results, "
            f"found {len(ig['enrichment'])}"
        )

        assert len(ig["undefined"]) == 500, (
            f"{name}: undefined array is not length 500."
        )

        assert len(ig["completeness_error"]) == 500, (
            f"{name}: completeness_error is not length 500."
        )

        assert len(ig["steps_used"]) == 500, (
            f"{name}: steps_used is not length 500."
        )

        assert len(ig["signed_attribution"]) == 500, (
            f"{name}: signed_attribution is not length 500."
        )


print(
    "✓ all 30 models have extraction "
    "(final + best) and IG results present."
)

print(
    "✓ safe to compute endpoints and "
    "confirmatory statistics."
)


# ================================================================
# FROZEN ENDPOINT EXTRACTION
# ================================================================

def endpoints_row(o, ck):

    name = (
        f"rho_{o['rho']:.1f}_rep_{o['replicate']}"
    )

    path = (
        LOGIT_DIR
        / f"{name}__{ck}.npz"
    )

    with np.load(path) as z:

        e = ap.model_endpoints(
            ev_y,
            z["z_clean"],
            z["z_hair"],
            z["z_reversed"],
            z["z_matched"],
            z["z_neutral"],
        )

        row = {
            "model": name,
            "rho": o["rho"],
            "replicate": o["replicate"],
            "checkpoint": ck,
            **e,
        }

        clean_pred = (
            z["z_clean"] >= 0
        ).astype(int)

    return row, clean_pred


# ================================================================
# COMPUTE ENDPOINTS FOR ALL 30 MODELS
# ================================================================

rows_final = []
rows_best = []

clean_pred_rho0_final = {}


for o in RUN_ORDER:

    # ------------------------------------------------------------
    # Primary final-epoch-15 checkpoint
    # ------------------------------------------------------------

    r_f, pred_f = endpoints_row(
        o,
        "final"
    )

    # ------------------------------------------------------------
    # Best-eval-BA sensitivity checkpoint
    # ------------------------------------------------------------

    r_b, _ = endpoints_row(
        o,
        "best"
    )

    name = (
        f"rho_{o['rho']:.1f}_rep_{o['replicate']}"
    )

    # ------------------------------------------------------------
    # XAI results
    # ------------------------------------------------------------

    ig_path = (
        IG_DIR
        / f"{name}.npz"
    )

    with np.load(ig_path) as ig:

        undefined = (
            ig["undefined"]
            .astype(bool)
        )

        enrichment = (
            ig["enrichment"]
            .astype(float)
        )

        defined_enrichment = np.where(
            undefined,
            np.nan,
            enrichment
        )

        r_f["xai_enrichment"] = float(
            np.nanmean(
                defined_enrichment
            )
        )

        r_f["xai_undefined_fraction"] = float(
            np.mean(undefined)
        )

    rows_final.append(
        r_f
    )

    rows_best.append(
        r_b
    )

    # ------------------------------------------------------------
    # rho=0 clean predictions for Case F
    # ------------------------------------------------------------

    if float(o["rho"]) == 0.0:

        clean_pred_rho0_final[
            int(o["replicate"])
        ] = pred_f


# ================================================================
# DATAFRAMES
# ================================================================

df_final = pd.DataFrame(
    rows_final
)

df_best = pd.DataFrame(
    rows_best
)

assert len(df_final) == 30, (
    f"Expected 30 final-checkpoint rows, "
    f"found {len(df_final)}"
)

assert len(df_best) == 30, (
    f"Expected 30 best-checkpoint rows, "
    f"found {len(df_best)}"
)


# ================================================================
# SAVE PER-MODEL ENDPOINT TABLES
# ================================================================

df_final.to_csv(
    RESULTS
    / "model_endpoints_final_checkpoint.csv",
    index=False
)

df_best.to_csv(
    RESULTS
    / "model_endpoints_best_checkpoint.csv",
    index=False
)

print(
    f"per-model endpoints written "
    f"({len(df_final)} models, final checkpoint | "
    f"{len(df_best)} models, best checkpoint)"
)


# ================================================================
# CASE F — CHURN REFERENCE
# ================================================================

P0 = np.array(
    [
        clean_pred_rho0_final[r]
        for r in sorted(
            clean_pred_rho0_final
        )
    ]
)

assert P0.shape[0] == 5, (
    f"Expected 5 rho=0 replicates, "
    f"found {P0.shape[0]}"
)

assert P0.shape[1] == len(ev_y), (
    f"rho=0 prediction length mismatch: "
    f"{P0.shape[1]} vs {len(ev_y)}"
)

R = ap.churn_reference(
    P0
)

print(
    f"churn reference R (Case F) = {R:.5f}"
)


# ================================================================
# PRIMARY CONFIRMATORY ANALYSIS
# ================================================================

primary = ap.primary_analysis(
    df_final,
    n_perm=ap.N_PERM,
    R=R
)


# ================================================================
# SAVE PRIMARY ANALYSIS
# ================================================================

with open(
    RESULTS / "PRIMARY_ANALYSIS_RESULT.json",
    "w"
) as f:

    json.dump(
        primary,
        f,
        indent=2,
        default=str
    )


# ================================================================
# REPORT CONFIRMATORY RESULTS
# ================================================================

print(
    "\n"
    + "=" * 72
)

print(
    "STAGE 4 - CONFIRMATORY ANALYSIS"
)

print(
    "=" * 72
)

h1 = (
    primary["confirmatory"]
    ["H1_delta_z_hair"]
)

h2 = (
    primary["confirmatory"]
    ["H2_reversed_BA_degradation_D"]
)

print(
    f"H1  delta_z_hair vs rho    : "
    f"spearman={h1['spearman']:.4f}  "
    f"p_holm={h1['p_holm']:.5f}  "
    f"reject={h1['reject']}"
)

print(
    f"H2  reversed BA degrade    : "
    f"spearman={h2['spearman']:.4f}  "
    f"p_holm={h2['p_holm']:.5f}  "
    f"reject={h2['reject']}"
)


# ================================================================
# CASE F
# ================================================================

if "case_F" in primary:

    print(
        f"Case F: "
        f"{primary['case_F']['classification']}  "
        f"(C={primary['case_F']['C']:.4f}, "
        f"CI={primary['case_F']['ci95']}, "
        f"R={primary['case_F']['R']:.4f})"
    )


# ================================================================
# SECONDARY DESCRIPTIVE ANALYSES
# ================================================================

print(
    "\nSecondary (descriptive):"
)

for k, v in (
    primary[
        "secondary_descriptive"
    ].items()
):

    print(
        f"  {k:22s} "
        f"spearman={v['spearman']:.4f}  "
        f"p_raw={v['p_raw_descriptive']:.5f}"
    )


# ================================================================
# RHO=1 VS RHO=0 DESCRIPTIVE CONTRASTS
# ================================================================

print(
    "\nrho=1 vs rho=0 contrasts (descriptive):"
)

for k, v in (
    primary[
        "rho1_minus_rho0_descriptive"
    ].items()
):

    print(
        f"  {k:16s} "
        f"diff={v['rho1_minus_rho0']:.4f}  "
        f"95% CI="
        f"{[round(x, 4) for x in v['ci95']]}"
    )


# ================================================================
# BEST-CHECKPOINT SENSITIVITY
# ================================================================

r_sens, p_sens = ap.permutation_spearman(
    df_best["rho"],
    df_best["delta_z_hair"],
    n_perm=ap.N_PERM,
    seed=ap.PRIMARY_PERM_SEED
)

print(
    "\nBest-eval-BA checkpoint sensitivity: "
    f"spearman={r_sens:.4f}  "
    f"p={p_sens:.5f}"
)

print(
    "Primary final-epoch-15 H1: "
    f"spearman={h1['spearman']:.4f} / "
    f"p_holm={h1['p_holm']:.5f}"
)


# ================================================================
# SAVE STAGE 4 SUMMARY
# ================================================================

save_record(
    "stage4_summary.json",
    {
        "H1": h1,
        "H2": h2,
        "R": R,
        "case_F": primary.get("case_F"),

        "best_checkpoint_sensitivity": {
            "spearman": r_sens,
            "p": p_sens,
        },
    }
)


# ================================================================
# FINAL STATUS
# ================================================================

print(
    f"\n✓ Full results saved under {RESULTS}."
)

print(
    "✓ Stage 4 complete."
)

print(
    "✓ Commit this notebook version to keep "
    "the analysis outputs permanently."
)

Evaluation images: 4713
✓ all 30 models have extraction (final + best) and IG results present.
✓ safe to compute endpoints and confirmatory statistics.
per-model endpoints written (30 models, final checkpoint | 30 models, best checkpoint)
churn reference R (Case F) = 0.09007

STAGE 4 - CONFIRMATORY ANALYSIS
H1  delta_z_hair vs rho    : spearman=0.9866  p_holm=0.00002  reject=True
H2  reversed BA degrade    : spearman=0.9866  p_holm=0.00002  reject=True

Secondary (descriptive):
  delta_z_norm           spearman=0.9866  p_raw=0.00001
  delta_p                spearman=0.9866  p_raw=0.00001
  flip_fraction          spearman=0.9866  p_raw=0.00001
  AUROC_reversed         spearman=-0.9866  p_raw=0.00001
  D_AUROC                spearman=0.9866  p_raw=0.00001
  BA_neutral_minus_reversed spearman=0.9866  p_raw=0.00001
  xai_enrichment         spearman=0.9843  p_raw=0.00001

rho=1 vs rho=0 contrasts (descriptive):
  delta_z_hair     diff=10.5431  95% CI=[10.032, 11.0542]
  D                dif

### After Stage 4
Commit this version. `PRIMARY_ANALYSIS_RESULT.json` and the two `model_endpoints_*.csv` files under `results/` are your complete, pre-registered result set - write the paper's results section directly from these, in the format your Case A-F decision tree already specifies.